In [ ]:
import os, sys

# Run as if from the project root, regardless of where this notebook lives
if os.path.basename(os.getcwd()) == "notebook":
    os.chdir("..")
sys.path.insert(0, os.getcwd())

In [1]:
import numpy as np
import pickle
import re
from collections import defaultdict, Counter

import pandas as pd
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score
from sklearn.utils import shuffle


FEATURES_DIR = "./extracted_features"
LANGUAGES = ["en_us", "de_de", "es_419"]



In [3]:
def simple_g2p(text):
    
    
    text = text.lower().strip()
    text = re.sub(r"[^a-zäöüßàéèêíóúâêîôûçñ ]", "", text)

    consonants = set("bcdfghjklmnpqrstvwxyz")
    vowels = set("aeiou")

    phonemes = []

    for ch in text:
        if ch in consonants:
            phonemes.append(ch)
        elif ch in vowels:
            phonemes.append(ch.upper())

    return phonemes



PHONEME_FEATURES = {
    'p': {'voicing': 'voiceless', ' ': 'stop'},
    'b': {'voicing': 'voiced', 'manner': 'stop'},
    't': {'voicing': 'voiceless', 'manner': 'stop'},
    'd': {'voicing': 'voiced', 'manner': 'stop'},
    'k': {'voicing': 'voiceless', 'manner': 'stop'},
    'g': {'voicing': 'voiced', 'manner': 'stop'},

    'f': {'voicing': 'voiceless', 'manner': 'fricative'},
    'v': {'voicing': 'voiced', 'manner': 'fricative'},
    's': {'voicing': 'voiceless', 'manner': 'fricative'},
    'z': {'voicing': 'voiced', 'manner': 'fricative'},
    'h': {'voicing': 'voiceless', 'manner': 'fricative'},

    'm': {'voicing': 'voiced', 'manner': 'nasal'},
    'n': {'voicing': 'voiced', 'manner': 'nasal'},

    'l': {'voicing': 'voiced', 'manner': 'liquid'},
    'r': {'voicing': 'voiced', 'manner': 'liquid'},

    'w': {'voicing': 'voiced', 'manner': 'glide'},
    'j': {'voicing': 'voiced', 'manner': 'glide'},
}

def extract_phonological_features(text):
    phonemes = simple_g2p(text)

    if not phonemes:
        return None

    all_feats = defaultdict(list)

    for p in phonemes:
        if p in PHONEME_FEATURES:
            for k, v in PHONEME_FEATURES[p].items():
                all_feats[k].append(v)

    if not all_feats:
        return None

    return {
        feat: Counter(vals).most_common(1)[0][0]
        for feat, vals in all_feats.items()
    }



In [4]:



def load_features(file_path):
    with open(file_path, "rb") as f:
        return pickle.load(f)

def build_dataset(features, layer_idx=6):
    X, y_dict = [], defaultdict(list)

    for sample in features:
        hidden = sample["hidden_states"][layer_idx]

        # (T, D) → mean pooling
        emb = hidden.mean(axis=1).squeeze()

        text = sample.get("transcription", "")

        feats = extract_phonological_features(text)

        if feats is None:
            continue

        X.append(emb)

        for k, v in feats.items():
            y_dict[k].append(v)

    X = np.array(X)

    return X, y_dict


def train_probe(X_train, y_train):
    if len(set(y_train)) < 2:
        return None

    model = LogisticRegression(max_iter=1000, class_weight="balanced")
    model.fit(X_train, y_train)
    return model


### EXPERIMENT 1: CROSS-LINGUAL TRANSFER

In [11]:

def cross_lingual(layer=6):

    train_data = load_features(f"{FEATURES_DIR}/en_us_features.pkl")
    X_train, y_train_dict = build_dataset(train_data, layer)

    results = []

    for feat_name, y_train in y_train_dict.items():

        probe = train_probe(X_train, y_train)
        if probe is None:
            continue

        for lang in ["de_de", "es_419"]:
            test_data = load_features(f"{FEATURES_DIR}/{lang}_features.pkl")
            X_test, y_test_dict = build_dataset(test_data, layer)

            if feat_name not in y_test_dict:
                continue

            y_test = y_test_dict[feat_name]

            min_len = min(len(X_test), len(y_test))
            acc = accuracy_score(
                y_test[:min_len],
                probe.predict(X_test[:min_len])
            )

            results.append([feat_name, lang, acc])

            print(f"{feat_name} → {lang}: {acc:.3f}")

    return results


cross_results = cross_lingual(layer=6)


voicing → de_de: 0.770
voicing → es_419: 0.640
manner → de_de: 0.630
manner → es_419: 0.430


### EXPERIMENT 2: LAYER ANALYSIS

In [6]:


def layer_analysis(layers=[0, 3, 6, 9, 12]):

    data = load_features(f"{FEATURES_DIR}/en_us_features.pkl")

    results = []

    for layer in layers:
        print(f"\nLayer {layer}")

        X, y_dict = build_dataset(data, layer)

        for feat_name, y in y_dict.items():

            split = int(0.8 * len(X))

            X_train, X_test = X[:split], X[split:]
            y_train, y_test = y[:split], y[split:]

            probe = train_probe(X_train, y_train)

            if probe:
                acc = accuracy_score(y_test, probe.predict(X_test))
                results.append([layer, feat_name, acc])
                print(f"{feat_name}: {acc:.3f}")

    return results


layer_results = layer_analysis()



Layer 0
voicing: 0.400
manner: 0.650

Layer 3
voicing: 0.500
manner: 0.700

Layer 6
voicing: 0.500
manner: 0.800

Layer 9
voicing: 0.500
manner: 0.800

Layer 12
voicing: 0.400
manner: 0.800


### EXPERIMENT 3: LANGUAGE COMPARISON

In [7]:



def language_comparison(layer=6):

    results = []

    for lang in LANGUAGES:

        data = load_features(f"{FEATURES_DIR}/{lang}_features.pkl")
        X, y_dict = build_dataset(data, layer)

        for feat_name, y in y_dict.items():

            split = int(0.8 * len(X))

            X_train, X_test = X[:split], X[split:]
            y_train, y_test = y[:split], y[split:]

            probe = train_probe(X_train, y_train)

            if probe:
                acc = accuracy_score(y_test, probe.predict(X_test))
                results.append([lang, feat_name, acc])

                print(f"{lang} {feat_name}: {acc:.3f}")

    return results



lang_results = language_comparison(layer=6)


en_us voicing: 0.500
en_us manner: 0.800
de_de voicing: 1.000
de_de manner: 0.500
es_419 manner: 0.300


#### saving results

In [8]:

all_results = {
    "cross_lingual": cross_results,
    "layer_analysis": layer_results,
    "language_comparison": lang_results
}

with open(f"{FEATURES_DIR}/results.pkl", "wb") as f:
    pickle.dump(all_results, f)

print("Results saved to:", FEATURES_DIR)



Results saved to: ./extracted_features
